# Train and segment a tiled area

This notebook walks through a tiled end-to-end tutorial:

1. build tiles from the input rasters
2. attach training crowns to the intersecting tiles
3. build SAM training chips
4. train a SAM decoder
5. inspect SAM proposal outputs
6. build StarDist training records
7. train StarDist
8. run segmentation across all prepared tiles

This is a tutorial for understanding and exercising the tiled workflow.
It is not a recommended evaluation setup.
For real model assessment, keep separate training and held-out areas.


## Inputs

This notebook assumes the clipped rasters are already in the repo `data/` directory.

You still need to provide a crown-label file for training.
That label file can cover only part of the tiled area.


In [1]:
import sys
from pathlib import Path

import geopandas as gpd
import rasterio
from shapely.geometry import box


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "samstars").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current working directory.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from samstars.data import build_scale_stats_from_proposal_records, build_tiles
from samstars.segmentation import run_segmentation
from samstars.training.chips import build_training_chips
from samstars.training.records import build_stardist_training_records
from samstars.training.sam_training import train_decoder
from samstars.training.stardist_training import train_stardist_model


/Users/iosefa/miniforge3/envs/samstars/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
bioimageio_utils.py (2): pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


## Set paths

Update the label and checkpoint paths for your own setup.


In [21]:
data_dir = repo_root / "data"

img = data_dir / "img_clipped.tif"
chm = data_dir / "chm_clipped.tif"
den = data_dir / "den_clipped.tif"
crowns = data_dir / "crowns_training_smoothed_mild.gpkg"
sam_base_ckpt = repo_root / "sam/sam_vit_l_0b3195.pth"
work_dir = data_dir / "train_and_segment_tiled_area_run"
device = "mps"

tiles_dir = work_dir / "tiles"
training_chips_dir = work_dir / "training_chips"
training_run_dir = work_dir / "training_run"
records_dir = work_dir / "stardist_training_records"
models_dir = work_dir / "models"
inference_dir = work_dir / "segmentation_run"
sam_ckpt = models_dir / "sam_crowns_decoder.pth"
stardist_model = models_dir / "stardist"

print("repo_root:", repo_root)
print("data_dir:", data_dir)


repo_root: /Users/iosefa/repos/sam
data_dir: /Users/iosefa/repos/sam/data


## Set tutorial parameters

These are explicit on purpose. Do not treat them as fixed project defaults. Change them, rerun the notebook, and compare the intermediate proposal outputs before committing to StarDist training.

`rgb_bands` controls which three image bands are written into each tile. For a 3-band raster, keep `(1, 2, 3)`. For a multiband raster, choose the three bands you want the full workflow to use.


In [16]:
rgb_bands = (1, 2, 3)

tile_size_m = 500.0
tile_overlap_m = 20.0

chip_size = 512
chip_stride = 384

sam_epochs = 3
stardist_epochs = 10
val_fraction = 0.2

crop_smooth_sigma = 2.0
smooth_sigma = 4.0
post_smooth_sigma = 0.0

stardist_input_key = "file_name"
use_aux = True
fast_mode = False


## Build tiles

This creates the prepared tile layout used by the tiled workflow.


In [5]:
build_tiles(
    img=img,
    chm=chm,
    den=den,
    rgb_bands=rgb_bands,
    out=tiles_dir,
    tile=tile_size_m,
    overlap=tile_overlap_m,
    skip_existing=True,
)


Tiling:   0%|                                                                  | 0/4 [00:00<?, ?tile/s]function_base.py (4824): Warning: 'partition' will ignore the 'mask' of the MaskedArray.
cost_surface.py (137): No SLIC provided – cost built from 3 terms only.
_base.py (248): Precomputed sparse input was not sorted by row values. Use the function sklearn.neighbors.sort_graph_by_row_values to sort the input by row values, with warn_when_not_sorted=False to remove this warning.
Tiling:  25%|██████████████▌                                           | 1/4 [00:02<00:08,  2.81s/tile]function_base.py (4824): Warning: 'partition' will ignore the 'mask' of the MaskedArray.
cost_surface.py (137): No SLIC provided – cost built from 3 terms only.
_base.py (248): Precomputed sparse input was not sorted by row values. Use the function sklearn.neighbors.sort_graph_by_row_values to sort the input by row values, with warn_when_not_sorted=False to remove this warning.
Tiling:  50%|███████████████████

✓ Tiles written to /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/tiles


## Attach training crowns to intersecting tiles

This clips the training polygons into each tile that overlaps the labeled area.
Only those tiles contribute supervised training data.


In [6]:
crowns_gdf = gpd.read_file(crowns)
labeled_tiles = []

for tile_dir in sorted(p for p in tiles_dir.iterdir() if p.is_dir() and p.name.startswith("tile_")):
    img_path = tile_dir / "img.tif"
    if not img_path.exists():
        continue

    with rasterio.open(img_path) as src:
        tile_bounds = box(*src.bounds)
        tile_crs = src.crs

    tile_crowns = crowns_gdf
    if tile_crowns.crs != tile_crs:
        tile_crowns = tile_crowns.to_crs(tile_crs)

    tile_crowns = tile_crowns[tile_crowns.intersects(tile_bounds)].copy()
    if tile_crowns.empty:
        continue

    tile_crowns["geometry"] = tile_crowns.geometry.buffer(0)
    tile_crowns["geometry"] = tile_crowns.geometry.intersection(tile_bounds)
    tile_crowns = tile_crowns[~tile_crowns.geometry.is_empty]
    if tile_crowns.empty:
        continue

    out_path = tile_dir / "crowns.gpkg"
    tile_crowns.to_file(out_path, driver="GPKG")
    labeled_tiles.append(tile_dir.name)

print("labeled tiles:", labeled_tiles)
print("count:", len(labeled_tiles))


labeled tiles: ['tile_000_001', 'tile_001_001']
count: 2


## Build SAM training chips


In [7]:
build_training_chips(
    tiles_dir=tiles_dir,
    out_dir=training_chips_dir,
    chip_size=chip_size,
    stride=chip_stride,
)


Skipping tile_000_000: missing img.tif, crowns.gpkg, or chm.tif
tile_000_001: wrote 153 training chips
Skipping tile_001_000: missing img.tif, crowns.gpkg, or chm.tif
tile_001_001: wrote 8 training chips
Built 161 training chips in /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/training_chips


161

## Train models

This section first trains the SAM decoder, then creates and inspects proposal rasters, and only then trains StarDist.


In [8]:
train_decoder(
    chip_dir=training_chips_dir,
    ckpt=sam_base_ckpt,
    out_path=sam_ckpt,
    epochs=sam_epochs,
    device=device,
)

training_chips: epochs=3 (iters/epoch=161)
training_chips epoch 1/3 loss=0.5182
training_chips epoch 2/3 loss=0.3503
training_chips epoch 3/3 loss=0.3005
✓ weights → /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/models/sam_crowns_decoder.pth


In [9]:
proposal_result = run_segmentation(
    tiles_dir=tiles_dir,
    out_dir=training_run_dir,
    sam_ckpt=sam_ckpt,
    device=device,
    run_sam=True,
    run_stardist=False,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
    fast=fast_mode,
)

seeds in chip: 3518
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.11557551e-06 8.69005458e-01
 9.93547738e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.22819074e-06 8.59956285e-01
 9.87344265e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 9.15049553e-01
 9.98059750e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 9.27026110e-01
 9.99990523e-01]
seed prob pcts: [1.58057983e-10 3.43174179e-07 4.12076542e-06 8.65363877e-01
 9.64094698e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.22819074e-06 8.94965277e-01
 9.93713796e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.21389416e-06 8.18944156e-01
 9.92827952e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 9.15114717e-01
 9.99764800e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 8.79162332e-01
 9.75746572e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 8.94908934e-01
 9.99992788e-01]
seed prob pcts: [1.16361983e-10 1.893278

proposals.py (105): invalid value encountered in divide


prob_s pcts: [0.         0.00123952 0.42178355 0.73096829 0.90415215]
seeds in chip: 994
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.22819074e-06 9.27315354e-01
 9.99992788e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06623238e-06 8.94964649e-01
 9.99866903e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 9.18691519e-01
 9.99857485e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.11557551e-06 9.14305691e-01
 9.99992788e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.21389416e-06 5.86212431e-01
 9.70603287e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 8.77819592e-01
 9.83237028e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.35975574e-06 7.26633427e-01
 9.40530539e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.17138347e-06 9.27315354e-01
 9.99992788e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 9.27315354e-01
 9.99992430e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.18549087e-06 7.7

proposals.py (105): invalid value encountered in divide


prob_s pcts: [0.00000000e+00 2.32834871e-08 1.28801996e-01 6.09494450e-01
 8.65761578e-01]
seeds in chip: 970
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.22962676e-06 8.90363982e-01
 9.95711148e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.18833120e-06 8.78939275e-01
 9.95384097e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.21389416e-06 7.93835310e-01
 9.86794114e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.35975574e-06 8.40744079e-01
 9.80853558e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.24255097e-06 6.30692481e-01
 8.64016175e-01]


proposals.py (105): invalid value encountered in divide


prob_s pcts: [0.         0.06837799 0.53970146 0.75773682 0.88711429]
seeds in chip: 445
seed prob pcts: [2.14694901e-10 2.25108224e-07 3.63940494e-06 9.53470745e-01
 9.98285294e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 8.37900814e-01
 9.83982623e-01]
seed prob pcts: [1.58057983e-10 3.26821691e-07 4.44372654e-06 8.68898961e-01
 9.79038119e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.17138347e-06 9.20492488e-01
 9.99910891e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.23370864e-06 7.26474264e-01
 9.36857760e-01]
seed prob pcts: [1.58057983e-10 2.91212515e-07 3.94186554e-06 6.08986784e-06
 2.26323724e-01]
prob_s pcts: [0.         0.00991132 0.42817565 0.69541429 0.91426486]
✓ wrote proposal outputs (4 records)


proposals.py (105): invalid value encountered in divide


## Inspect SAM proposal outputs

Inspect the proposal rasters before continuing.
If the proposals are poor, adjust the parameters above and rerun this section before training StarDist.


In [ ]:
proposal_result


In [ ]:
proposal_rasters_dir = proposal_result["proposals_root"] / "rasters"
sorted(p.name for p in proposal_rasters_dir.glob("*.tif"))[:40]


## Build StarDist training records


In [10]:
train_records, val_records = build_stardist_training_records(
    proposal_records=proposal_result["proposal_records"],
    tiles_dir=tiles_dir,
    out_dir=records_dir,
    val_fraction=val_fraction,
)

print("train records:", len(__import__('json').load(open(train_records))))
print("val records:", len(__import__('json').load(open(val_records))))


✓ wrote /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/stardist_training_records/train_records.json (1 records)
✓ wrote /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/stardist_training_records/val_records.json (1 records)
train records: 1
val records: 1


In [11]:
scale_stats = work_dir / "training_scale_stats.json"

build_scale_stats_from_proposal_records(
    proposal_records=proposal_result["proposal_records"],
    out=scale_stats,
    percentiles=(2.0, 98.0),
    chm_key="chm",
    nodata_values=(-9999.0,),
)


{'percentiles': [2.0, 98.0],
 'chm_01': {'lo': 0.0, 'hi': 6.8467930316924965},
 'source_proposal_records': '/Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/training_run/proposals/proposal_records.json',
 'records_total': 4,
 'records_missing_or_unreadable_chm': 0}

In [ ]:
stardist_model = models_dir / "stardist_prob_tiled"

train_stardist_model(
    train_records=train_records,
    val_records=val_records,
    out_dir=stardist_model.parent,
    model_name=stardist_model.name,
    epochs=stardist_epochs,
    input_key=stardist_input_key,
    use_aux=use_aux,
    scale_stats_path=scale_stats,
)


epochs=10 (iters/epoch=1)
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.149439, nms_thresh=0.3.
Epoch 1/10


base_model.py (203): output path for model already exists, files may be overwritten: /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/models/stardist


100/100 ━━━━━━━━━━━━━━━━━━━━ 27s 240ms/step - dist_dist_iou_metric: 0.2315 - dist_loss: 6.2915 - dist_relevant_mae: 6.2909 - dist_relevant_mse: 65.5745 - loss: 1.5247 - prob_kld: 0.2132 - prob_loss: 0.2664 - val_dist_dist_iou_metric: 0.2833 - val_dist_loss: 2.6069 - val_dist_relevant_mae: 2.6060 - val_dist_relevant_mse: 9.2025 - val_loss: 0.5838 - val_prob_kld: 0.0620 - val_prob_loss: 0.0624 - learning_rate: 3.0000e-04
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 24s 236ms/step - dist_dist_iou_metric: 0.3686 - dist_loss: 4.8443 - dist_relevant_mae: 4.8434 - dist_relevant_mse: 37.6208 - loss: 1.1387 - prob_kld: 0.1162 - prob_loss: 0.1698 - val_dist_dist_iou_metric: 0.2985 - val_dist_loss: 2.0761 - val_dist_relevant_mae: 2.0751 - val_dist_relevant_mse: 6.9742 - val_loss: 0.4596 - val_prob_kld: 0.0440 - val_prob_loss: 0.0444 - learning_rate: 3.0000e-04
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 24s 241ms/step - dist_dist_iou_metric: 0.3703 - dist_loss: 4.7803 - dist_relevant_mae: 4.7794 - dist_re

## Run tiled segmentation

This runs inference across all prepared tiles.


In [22]:
stardist_model = models_dir / "stardist"

result = run_segmentation(
    tiles_dir=tiles_dir,
    out_dir=training_run_dir,
    sam_ckpt=sam_ckpt,
    stardist_model=stardist_model,
    stardist_input_key=stardist_input_key,
    device=device,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
    fast=fast_mode,
    run_sam=False
)

bioimageio_utils.py (2): pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
2026-04-19 22:58:09.516977: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Max
2026-04-19 22:58:09.516991: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-04-19 22:58:09.516995: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 48.00 GB
2026-04-19 22:58:09.517008: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-19 22:58:09.517019: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> p

Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.178668, nms_thresh=0.3.
Using effective thresholds: prob_thresh=0.17866772413253784, nms_thresh=0.3.
✓ wrote 1539 crown polygons to /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/training_run/crowns/crowns.gpkg


## Final output


In [20]:
inference_dir / "crowns" / "crowns.gpkg"


PosixPath('/Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/segmentation_run/crowns/crowns.gpkg')